# Spark Fundamentals — Data Cleaning, Transformation & Aggregation


**Objective:** Understand Spark fundamentals and perform data cleaning, transformation, and aggregation using DataFrames.

**Dataset used:** `sales_data.csv` 

This notebook walks through each step of the objective:
1. Limitations of MapReduce and advantages of Spark
2. Spark DataFrame concepts and immutability
3. Data cleaning (duplicates, null values)
4. Filtering conditions (age range, category, region)
5. Aggregation functions (count, sum, avg, min, max)
6. Grouping with `groupBy` and conditions on aggregated results
7. Wide transformations and shuffle operations
8. Schema modification (casting, renaming)
9. Handling inconsistent data
10. A complete data processing pipeline

Also this notebook answers these questions:
1. What are the key limitations of traditional MapReduce that make Spark a preferred choice for modern big data processing? 
2. Explain how Spark uses In-Memory Computing to speed up iterative machine learning algorithms compared to disk-based systems. 
3. Write a code snippet to remove all duplicate rows from a DataFrame based on a specific set of columns: user_id and transaction_date. 
4. Given a DataFrame df_sales, write a query to filter for rows where the region is 'West' and then group by product_category to find the average sale_amount. 
5. What is the difference between .na.drop() and .na.fill()? Provide a code example of filling null values in a status column with the string 'Unknown'. 
6. Write a query to find the total count of records for each city in a DataFrame, but only for cities where the count is greater than 100. 
7. How does the immutability of Spark DataFrames affect how you perform "data cleaning" steps like dropping columns or renaming them? 
8. Write a Spark command to filter a dataset for rows where the age is between 18 and 30 (inclusive) and the subscription is 'Premium'. 
9. When cleaning a dataset, why is it often better to handle null values before performing mathematical aggregations like sum() or avg()? 
10. Write the code to revise a column named raw_timestamp by casting it to a TimestampType and renaming it to event_time. 
11. Explain the "Shuffle" process that occurs during a grouping operation. Why is it considered a wide transformation? 
12. Write a code snippet that identifies and removes rows where the email column contains null values OR the username is an empty string. 
13. How do you use the .agg() function to calculate multiple statistics at once, such as the min, max, and mean of the price column? 
14. In the context of cleaning a dataset, what is the risk of using inferSchema=true when your source data contains messy or inconsistent date formats? 
15. Write a final processing pipeline that: 
- Filters out duplicates. 
- Fills null prices with 0. 
- Groups by store_id to calculate total revenue. 

Each section includes PySpark code, query results, and brief insights.

## 1. Limitations of MapReduce and Advantages of Spark

**Limitations of traditional MapReduce**
- **Heavy disk dependency** :- data generated after each Map and Reduce phase is stored on HDFS, leading to significant read/write overhead in multi-step workflows.
- **Not ideal for iterative processing** :- ML and graph algorithms reload the same data from disk on every iteration.
- **Restricted processing model** :- tasks need to fit into the Map and Reduce pattern, which can make complex data workflows difficult to design.
- **Higher execution latency** :- primarily built for batch processing, making real-time analysis and interactive querying challenging.
- **Limited programming flexibility** :- developers mainly work with map and reduce operations, resulting in more code for advanced processing tasks.

**Advantages of Spark over MapReduce**
- **In-memory processing** :- frequently used data can be retained in memory, significantly reducing processing time for repeated operations.
- **DAG execution engine** :- Spark builds an optimized Directed Acyclic Graph instead of rigid Map→Reduce stages.
- **Comprehensive APIs** :- supports RDDs, DataFrames, and SQL, along with a wide range of built-in transformations and actions.
- **Optimization through lazy execution** :- operations are planned and optimized before running, improving overall performance.
- **Fault tolerance** :- Spark tracks transformation history, allowing lost data partitions to be rebuilt when failures occur.

## 2. Spark DataFrame Concepts & Immutability

A **DataFrame** is a distributed and immutable collection of data arranged into named columns, similar to a database table. Key concepts include:

- **Immutability** — applying a transformation does not change the existing DataFrame.Instead, Spark creates a new DataFrame, so any modifications should be stored in a new or updated variable.
- **Lazy evaluation** — transformations such as (`filter`, `select`, `withColumn`) only define the execution plan.The operations are performed only when an **action** like (`show`, `count`, `collect`) is called.

Creating a Spark session and loading the dataset:-

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import TimestampType

# creating a spark session 
spark = (SparkSession.builder
         .appName("Week5_Spark_Fundamentals")
         .master("local[*]")
         .getOrCreate())

# setting the log level to ERROR
spark.sparkContext.setLogLevel("ERROR")

In [2]:
# Load the dataset
# inferSchema=True, this tells Spark to automatically detect the data type of each column by examining the data.
# to run this code, make sure you have the sales_data.csv file in the data folder.

df = spark.read.csv("../data/sales_data.csv", header=True, inferSchema=True)

print("Rows:", df.count(), "| Columns:", len(df.columns))
df.printSchema()

Rows: 2320 | Columns: 14
root
 |-- user_id: string (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- region: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- sale_amount: double (nullable = true)
 |-- price: double (nullable = true)
 |-- status: string (nullable = true)
 |-- city: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- subscription: string (nullable = true)
 |-- email: string (nullable = true)
 |-- username: string (nullable = true)
 |-- raw_timestamp: timestamp (nullable = true)
 |-- store_id: string (nullable = true)



In [3]:
# Preview the data 
# truncate=False, this shows the complete value of each column without cutting it off.
df.show(10, truncate=False)

+-------+----------------+------+----------------+-----------+-------+--------+---------+---+------------+-----------------+---------+-------------------+--------+
|user_id|transaction_date|region|product_category|sale_amount|price  |status  |city     |age|subscription|email            |username |raw_timestamp      |store_id|
+-------+----------------+------+----------------+-----------+-------+--------+---------+---+------------+-----------------+---------+-------------------+--------+
|U1863  |2025-02-08      |West  |Electronics     |109.34     |406.56 |Inactive|Delhi    |36 |Free        |u1863@yahoo.com  |NULL     |2025-02-08 15:48:09|S022    |
|U1099  |2024-05-05      |North |Furniture       |1174.05    |560.49 |NULL    |Mumbai   |42 |Free        |NULL             |NULL     |2024-05-05 01:47:13|S021    |
|U1337  |2024-11-05      |North |Furniture       |4982.25    |402.99 |NULL    |Mumbai   |45 |Premium     |u1337@gmail.com  |user_1337|2024-11-05 17:24:03|S003    |
|U1634  |2024-02

**Demonstrating dataframe immutability** :- operations such as `df.drop(...)` create a new DataFrame instead of modifying the existing one. To keep the changes, the result must be assigned to a variable.

Below code demonstrates Spark DataFrame immutability:

- `df` remains unchanged (`username` is still present).
- `temp` is a new dataFrame where the `username` column has been removed.

In [4]:
temp = df.drop("username")          # returns a NEW DataFrame
print("Original columns still exist :", "username" in df.columns)
print("New DataFrame dropped it :", "username" in temp.columns)

Original columns still exist : True
New DataFrame dropped it : False


## 3. Data Cleaning — Duplicates & Null Values

- Firstly, we inspect how dirty the data is: count fully duplicated rows and nulls/empties per column.

In [5]:
total = df.count()
distinct = df.dropDuplicates().count()
print(f"Total rows      : {total}")
print(f"Distinct rows   : {distinct}")
print(f"Duplicate rows  : {total - distinct}")

Total rows      : 2320
Distinct rows   : 2200
Duplicate rows  : 120


- Now we check for the NULL or empty string counts per column

In [6]:
from pyspark.sql.functions import col, when, count

# Generate a report of NULL and empty-string counts for each column
null_report = df.select([
    count(when(col(c).isNull() | (col(c).cast("string") == ""), c)).alias(c)
    for c in df.columns
])
null_report.show(truncate=False)

+-------+----------------+------+----------------+-----------+-----+------+----+---+------------+-----+--------+-------------+--------+
|user_id|transaction_date|region|product_category|sale_amount|price|status|city|age|subscription|email|username|raw_timestamp|store_id|
+-------+----------------+------+----------------+-----------+-----+------+----+---+------------+-----+--------+-------------+--------+
|0      |0               |0     |0               |0          |184  |606   |0   |0  |0           |148  |150     |0            |0       |
+-------+----------------+------+----------------+-----------+-----+------+----+---+------------+-----+--------+-------------+--------+



 **Remove duplicates:-** `dropDuplicates()` removes rows that are completely identical when no columns are specified. If a subset of columns is provided, Spark retains the first occurrence of each unique combination and removes the remaining duplicates.

 Below code removes the duplicates:-
 - **Remove complete duplicates:** Calling `dropDuplicates()` without any arguments removes rows that are identical across all columns.
- **Remove duplicates using keys:** Passing specific columns to `dropDuplicates()` removes duplicate records based on those columns while keeping the first occurrence of each unique combination.

In [7]:
# Remove fully duplicate rows
df_noDuplicates = df.dropDuplicates()
print("After removing full duplicates:", df_noDuplicates.count())

# Remove duplicate records based on the combination of user_id and transaction_date
df_key_dup = df.dropDuplicates(["user_id", "transaction_date"])
print("Unique (user_id, transaction_date) rows:", df_key_dup.count())

After removing full duplicates: 2200
Unique (user_id, transaction_date) rows: 2191


**Handle null values:-** `.na.drop()` is used to remove rows containing null values, while `.na.fill()` replaces missing values with a specified value.

 In this dataset, the `price` column is read as a **string** because it contains empty strings, even when `inferSchema=True` is enabled. Converting the column to a numeric data type changes invalid or empty values into `null`, so we can fill them with numeric values.

- In this dataset we fill missing `price` values with `0` (example only; in production,we will choose an appropriate strategy based on business requirements)
- Filled missing `status` values with `Unknown`
- Droped rows missing an `email` OR with an empty `username` (invalid records)

In [8]:
# 'price' was read as a string because of empty values -- so cast to double first
df_clean = df_noDuplicates.withColumn("price", col("price").cast("double"))

# filled nulls in price with 0 
df_clean = df_clean.na.fill({"price": 0})

# filled null/empty strings in status as 'Unknown'
df_clean = df_clean.withColumn(
    "status",
    when((col("status").isNull()) | (col("status") == ""), "Unknown").otherwise(col("status"))
)

In [9]:
# Dropped rows missing an email OR with an empty username (invalid records)
df_clean = df_clean.filter(
    col("email").isNotNull() & (col("email") != "") & (col("username") != "")
)
print("Rows after dropping invalid email/username:", df_clean.count())

Rows after dropping invalid email/username: 1929


- Now we can see that we have removed the duplicates or the missing values from each column

In [10]:
from pyspark.sql.functions import col, when, count

null_report = df_clean.select([
    count(when(col(c).isNull() | (col(c).cast("string") == ""), c)).alias(c)
    for c in df_clean.columns
])
null_report.show(truncate=False)

+-------+----------------+------+----------------+-----------+-----+------+----+---+------------+-----+--------+-------------+--------+
|user_id|transaction_date|region|product_category|sale_amount|price|status|city|age|subscription|email|username|raw_timestamp|store_id|
+-------+----------------+------+----------------+-----------+-----+------+----+---+------------+-----+--------+-------------+--------+
|0      |0               |0     |0               |0          |0    |0     |0   |0  |0           |0    |0       |0            |0       |
+-------+----------------+------+----------------+-----------+-----+------+----+---+------------+-----+--------+-------------+--------+



## 4. Filtering Conditions (age range, category, region)

 **Filter Premium Customers Aged 18–30**
 
 In this example, the dataset is filtered to include only customers who:

- Are between **18 and 30 years of age** (inclusive).
- Have a **Premium** subscription.

In [11]:
# Age between 18 and 30 (inclusive) AND Premium subscription
premium_young = df_clean.filter(
    (col("age").between(18, 30)) & (col("subscription") == "Premium")
)
print("Premium customers aged 18-30:", premium_young.count())
premium_young.select("user_id", "age", "subscription", "region").show(10)

Premium customers aged 18-30: 117
+-------+---+------------+------+
|user_id|age|subscription|region|
+-------+---+------------+------+
|  U1338| 23|     Premium| South|
|  U1113| 18|     Premium|  West|
|  U1743| 25|     Premium| South|
|  U1688| 20|     Premium|  East|
|  U1175| 22|     Premium|  East|
|  U1745| 22|     Premium| North|
|  U1104| 26|     Premium| South|
|  U1547| 30|     Premium|  West|
|  U1564| 30|     Premium| South|
|  U1247| 25|     Premium|  West|
+-------+---+------------+------+
only showing top 10 rows


**Filter Electronics Sales from the West Region**

In this example, the DataFrame is filtered to include only transactions where:

- The **region** is **"West"**.
- The **product category** is **"Electronics"**.

In [12]:
# Region = 'West' and category = 'Electronics'
west_elec = df_clean.filter(
    (col("region") == "West") & (col("product_category") == "Electronics")
)
print("West + Electronics rows:", west_elec.count())
west_elec.select("region", "product_category", "sale_amount").show(10)

West + Electronics rows: 87
+------+----------------+-----------+
|region|product_category|sale_amount|
+------+----------------+-----------+
|  West|     Electronics|    1434.89|
|  West|     Electronics|    1527.04|
|  West|     Electronics|    3788.09|
|  West|     Electronics|    4106.95|
|  West|     Electronics|    1327.58|
|  West|     Electronics|     193.14|
|  West|     Electronics|      109.0|
|  West|     Electronics|    4898.42|
|  West|     Electronics|    3802.92|
|  West|     Electronics|    4561.98|
+------+----------------+-----------+
only showing top 10 rows


## 5. Aggregation Functions (count, sum, avg, min, max)

`.agg()` computes multiple statistics in one pass. Below we summarize `sale_amount` across the whole dataset.

In this example, aggregate functions are applied to the `sale_amount` column to generate key business metrics.

The following statistics are calculated:

- **Record Count** - Total number of records in the dataset.
- **Total Sales** - Sum of all sales amounts.
- **Average Sale** - Average sales amount, rounded to two decimal places.
- **Minimum Sale** - Lowest sales amount recorded.
- **Maximum Sale** - Highest sales amount recorded.

In [13]:
df_clean.agg(
    F.count("*").alias("record_count"),
    F.sum("sale_amount").alias("total_sales"),
    F.round(F.avg("sale_amount"), 2).alias("avg_sale"),
    F.min("sale_amount").alias("min_sale"),
    F.max("sale_amount").alias("max_sale")
).show()

+------------+------------------+--------+--------+--------+
|record_count|       total_sales|avg_sale|min_sale|max_sale|
+------------+------------------+--------+--------+--------+
|        1929|4782224.8199999975| 2479.12|   60.89| 4998.81|
+------------+------------------+--------+--------+--------+



## 6. Grouping with `groupBy` + Conditions on Aggregated Results


**Calculating Average Sales by Product Category in West region**

In this example, the dataset is first filtered to include only records from the **West** region. The data is then grouped by **product category**, and the average sales amount is calculated for each category.

The results are sorted in descending order of average sales amount, making it easier to identify the highest-performing product categories in the West region.

In [14]:
# Average sale_amount per product_category in the West region
sales_by_category = (df_clean
    .filter(col("region") == "West")
    .groupBy("product_category")
    .agg(F.round(F.avg("sale_amount"), 2).alias("avg_sale_amount"))
    .orderBy(F.col("avg_sale_amount").desc()))
sales_by_category.show()

+----------------+---------------+
|product_category|avg_sale_amount|
+----------------+---------------+
| Office Supplies|        2712.81|
|     Electronics|        2603.48|
|       Furniture|        2507.21|
|            Toys|        2433.52|
|        Clothing|        2392.08|
|       Groceries|        2373.88|
+----------------+---------------+



**Identifying cities with more than 100 records**

In this example, the dataset is grouped by **city** to count the number of records for each city.

After aggregation, only cities with **more than 100 records** are retained.The final output is sorted in descending order of record count.

In [15]:
# Count records per city, keep only cities with count > 100  
busy_cities = (df_clean
    .groupBy("city")
    .agg(F.count("*").alias("record_count"))
    .filter(col("record_count") > 100)
    .orderBy(F.col("record_count").desc()))
busy_cities.show()

+---------+------------+
|     city|record_count|
+---------+------------+
|   Mumbai|         457|
|    Delhi|         417|
|Bangalore|         372|
|  Chennai|         133|
|  Kolkata|         124|
|     Pune|         111|
+---------+------------+



## 7. Wide Transformations & Shuffle Operations

- **Narrow transformations** (`filter`, `map`, `select`) — each output partition is derived from a single input partition, so data remains within the same partition and no shuffle is required.
- **Wide transformations** (`groupBy`, `join`, `distinct`, `repartition`) — output partitions depend on data from multiple input partitions. Spark performs a **shuffle**, which redistributes data across the cluster by writing intermediate data, transferring it over the network, and repartitioning it based on the required key.

`groupBy` is considered a wide transformation because all records with the same grouping key must be brought together into the same partition before aggregation can be performed. The physical execution plan generated by `explain()` typically includes an **Exchange** (or `hashpartitioning`) operation, indicating that a shuffle has taken place.

In [16]:
# This is a wide transformation because Spark must shuffle data so that
# all records with the same region are brought into the same partition.
(df_clean.groupBy("region")
         .agg(F.sum("sale_amount").alias("total"))
         .explain())

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[region#19], functions=[sum(sale_amount#21)])
   +- Exchange hashpartitioning(region#19, 200), ENSURE_REQUIREMENTS, [plan_id=1292]
      +- HashAggregate(keys=[region#19], functions=[partial_sum(sale_amount#21)])
         +- HashAggregate(keys=[city#24, email#27, username#28, store_id#30, subscription#26, age#25, user_id#17, sale_amount#21, status#23, region#19, transaction_date#18, product_category#20, raw_timestamp#29, price#22], functions=[])
            +- Exchange hashpartitioning(city#24, email#27, username#28, store_id#30, subscription#26, age#25, user_id#17, sale_amount#21, status#23, region#19, transaction_date#18, product_category#20, raw_timestamp#29, price#22, 200), ENSURE_REQUIREMENTS, [plan_id=1288]
               +- HashAggregate(keys=[city#24, email#27, username#28, store_id#30, subscription#26, age#25, user_id#17, knownfloatingpointnormalized(normalizenanandzero(sale_amount#21)) AS sale_amoun

## 8. Schema Modification — Casting & Renaming

**Modify Data Types and Rename Columns**

In this example, the `raw_timestamp` column is converted from a **string** to the **TimestampType** data type, making it suitable for time-based analysis and operations.

After the conversion, we renamed the column to `event_time` to provide a more meaningful and descriptive name. Finally, the updated column values and the DataFrame schema are displayed to verify the changes.

In [17]:
# Changed raw_timestamp to TimestampType and renamed it to event_time
df_schema = (df_clean
    .withColumn("raw_timestamp", col("raw_timestamp").cast(TimestampType()))
    .withColumnRenamed("raw_timestamp", "event_time"))

df_schema.select("event_time").show(10, truncate=False)
df_schema.printSchema()

+-------------------+
|event_time         |
+-------------------+
|2024-08-25 01:33:13|
|2024-05-30 12:54:07|
|2025-05-22 02:30:48|
|2024-02-23 00:39:34|
|2025-01-23 06:21:03|
|2024-09-08 04:35:15|
|2024-12-14 16:03:12|
|2025-03-19 01:21:09|
|2024-04-18 15:48:12|
|2025-05-05 15:22:58|
+-------------------+
only showing top 10 rows
root
 |-- user_id: string (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- region: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- sale_amount: double (nullable = true)
 |-- price: double (nullable = false)
 |-- status: string (nullable = true)
 |-- city: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- subscription: string (nullable = true)
 |-- email: string (nullable = true)
 |-- username: string (nullable = true)
 |-- event_time: timestamp (nullable = true)
 |-- store_id: string (nullable = true)



## 9. Handling Inconsistent Data (nulls, empty values, schema issues)


**Q14: In the context of cleaning a dataset, what is the risk of using inferSchema=true when your source data contains messy or inconsistent date formats?**

- `inferSchema=True` is useful for automatically detecting column data types, but it may produce unexpected results when date or timestamp values have inconsistent formats. In such cases, Spark can interpret the column as a string or convert invalid values to `null`. A more reliable approach is to initially read date columns as strings and then explicitly convert them using `to_timestamp()` with the expected date format. This makes data quality issues easier to detect and handle.

Parsing date and time values with a predefined format ensures that timestamp columns are converted accurately. Instead of relying on automatic type inference, this approach explicitly converts the `raw_timestamp` column to the `TimestampType` data type using the expected format (`yyyy-MM-dd HH:mm:ss`).

In [18]:
# Explicit, safe timestamp parsing with a known format
df_safe = df.withColumn(
    "event_time",
    F.to_timestamp(col("raw_timestamp"), "yyyy-MM-dd HH:mm:ss")
)

# count any values that failed to parse (would become null)
bad = df_safe.filter(col("event_time").isNull() & col("raw_timestamp").isNotNull()).count()
print("Unparseable timestamps:", bad)
df_safe.select("raw_timestamp", "event_time").show(10, truncate=False)

Unparseable timestamps: 0
+-------------------+-------------------+
|raw_timestamp      |event_time         |
+-------------------+-------------------+
|2025-02-08 15:48:09|2025-02-08 15:48:09|
|2024-05-05 01:47:13|2024-05-05 01:47:13|
|2024-11-05 17:24:03|2024-11-05 17:24:03|
|2024-02-03 05:15:22|2024-02-03 05:15:22|
|2024-07-22 16:23:01|2024-07-22 16:23:01|
|2024-04-26 04:26:00|2024-04-26 04:26:00|
|2024-07-19 16:51:11|2024-07-19 16:51:11|
|2025-03-20 03:33:42|2025-03-20 03:33:42|
|2024-11-28 15:09:04|2024-11-28 15:09:04|
|2024-12-19 22:24:30|2024-12-19 22:24:30|
+-------------------+-------------------+
only showing top 10 rows


## 10. Complete Data Processing Pipeline

Combine cleaning and aggregation into one chained pipeline:
1. Remove duplicate rows
2. Fill null prices with 0
3. Group by `store_id` to calculate total revenue

In [19]:
pipeline = (df
    .dropDuplicates()                                  
    .withColumn("price", col("price").cast("double"))   
    .na.fill({"price": 0})                              
    .groupBy("store_id")                              
    .agg(F.round(F.sum("sale_amount"), 2).alias("total_revenue"),
         F.count("*").alias("transactions"))
    .orderBy(F.col("total_revenue").desc()))

pipeline.show(10)

+--------+-------------+------------+
|store_id|total_revenue|transactions|
+--------+-------------+------------+
|    S003|    277775.32|         104|
|    S023|    273446.37|         103|
|    S020|    246838.14|          91|
|    S022|    241236.92|          98|
|    S024|    238388.94|          92|
|    S014|    232475.63|          89|
|    S017|    230839.05|          92|
|    S001|    230281.17|          95|
|    S021|    223025.65|          93|
|    S007|    221698.72|          84|
+--------+-------------+------------+
only showing top 10 rows


## Insights on Data Processing & Transformations

- **In-memory processing improves performance:** Spark stores frequently used data in memory, reducing repeated disk access and making iterative workloads such as machine learning and multi-stage data pipelines much faster than traditional mapReduce.
- **Immutability influences DataFrame operations:** every transformation creates a new dataFrame instead of modifying the existing one. As a result, transformations should be reassigned or chained together, allowing Spark to optimize execution and recover data through lineage if failures occur.
- **Data cleaning should precede analysis:** removing duplicate records and handling missing values before performing aggregations such as `sum` or `avg` helps produce accurate and meaningful results.
- **Shuffle operations impact performance:** wide transformations, including `groupBy` and `join`, require Spark to redistribute data across partitions. This shuffle process involves network communication and disk operations, making it one of the most resource-intensive parts of Spark execution.
- **Explicit data types provide greater reliability:** while `inferSchema` is convenient, defining schemas manually or converting columns using functions such as `to_timestamp()` with a specified format results in more consistent data processing and makes invalid values easier to identify.

In [20]:
spark.stop()
print("Spark session stopped.")

Spark session stopped.
